In [ ]:
import dai
import pandas as pd

sql_test = "SELECT * FROM bigalpha_2026_stock_bar1m_test"
sql_cut = "SELECT * FROM bigalpha_2026_stock_bar1m_cut"

dates = pd.date_range(start='2024-09-01', end='2025-03-31', freq='D')
dates = [pd.to_datetime(i).strftime('%Y-%m-%d') for i in dates]

result = {}
for today in dates:
    df_test = dai.query(sql_test, filters={'date': [f'{today} 00:00:00', f'{today} 23:59:59']}).df()
    df_valid = dai.query(sql_cut, filters={'date': [f'{today} 00:00:00', f'{today} 23:59:59']}).df()

    if df_valid.empty:
        print(f'{today} 没有 valid 数据跳过')
        continue
    result[today] = {}
    cols = [i for i in df_test.columns if i not in ['date', 'instrument', 'instrument_id']]
    merge_df = pd.merge(df_test, df_valid, how='right', on=['date', 'instrument'], suffixes=['_test', '_valid'])
    for col in cols:
        merge_df['diff'] = (merge_df[f'{col}_test'] - merge_df[f'{col}_valid']).abs().mean()
        diff_df = merge_df[merge_df['diff']>0]
        if diff_df.empty:
            continue
        print(f'{today} {col} 有差异')
        result[today][col] = diff_df